# 01 — Test ib-gateway (connexion + handshake TWS API)

Smoke test du container `fxvol-ib-gateway`. Aucune dépendance runtime — ib-gateway est une **feuille** dans le graphe (cf. `docs/container_deps.md`), seul le compte IB paper et les secrets SSM sont nécessaires côté test.

**Couvre** :
1. Container UP + état du healthcheck Docker (`nc -z 127.0.0.1 4002`)
2. TCP probe direct sur `127.0.0.1:4002` depuis l'host (sans toucher à TWS API)
3. Secrets présents en env (`IB_USERID`, `IB_PASSWORD`, `VNC_PASSWORD`) — vérification non-révélante (longueur uniquement, **jamais** la valeur)
4. `IB.connect()` depuis `ib_insync` → `isConnected() == True`
5. `reqCurrentTime()` renvoie un timestamp serveur cohérent (±5s vs `datetime.now(UTC)`)
6. `disconnect()` propre + reconnect immédiat avec **autre** clientId → OK (preuve qu'on n'a pas laissé de session zombie)

**Préreq** :
- Container démarré : `docker compose --profile ib up -d ib-gateway`
- IBC login terminé : attendre que `docker logs fxvol-ib-gateway --tail 50` montre `Login has completed` (~60-90s après le `up`)
- Secrets chargés en env de la session courante : `.\scripts\load_secrets.ps1` (PowerShell host)
- `pip install ib_insync` (déjà dans `requirements.txt`)

**CLIENT_IDs réservés en prod** : `1` (orders), `2` (market-data engine), `3` (risk engine), `14` (notebooks de recherche legacy). Ce notebook utilise `199` (sandbox).

**Référence** : `docker-compose.yml` § `ib-gateway`, `infrastructure/ib-gateway/README.md`, `scripts/ib-gateway/SMOKE_PLAN.md`.

## Setup — imports + helpers

On instancie un seul `IB()` réutilisé entre les sections. `util.patchAsyncio()` est requis pour faire tourner `ib_insync` dans un kernel Jupyter (qui a déjà sa propre event loop).

In [1]:
import os
import socket
import subprocess
import time
from datetime import datetime, timezone

from ib_insync import IB, util

util.patchAsyncio()

HOST = "127.0.0.1"
PORT = 4002
CLIENT_ID = 199            # sandbox, hors plage prod (1, 2, 3) et research (14)
CLIENT_ID_BIS = 198        # pour le test reconnect avec un autre id
CONTAINER = "fxvol-ib-gateway"

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

ib = IB()  # réutilisé jusqu'à la section 6
print(f"target = {HOST}:{PORT}, clientId = {CLIENT_ID}")

target = 127.0.0.1:4002, clientId = 199


## 1. Container UP + healthcheck Docker

**Ce que tu dois voir** : container `running` ET healthcheck `healthy`. Un état `starting` (≤ 90s post-`up`) est acceptable — IBC est en cours de login. `unhealthy` = TCP probe échoue côté container, soit IB Gateway crashed, soit 2FA non validé.

Si tu vois `starting` depuis > 2 min : `docker logs fxvol-ib-gateway --tail 100` pour voir si IBC attend une push 2FA, si l'image est obsolète (`version is no longer supported`), ou si les credentials TWS_USERID/TWS_PASSWORD sont mal injectés.

In [2]:
print("== container state + healthcheck ==")

# Status général du container
out = subprocess.run(
    ["docker", "inspect", "-f", "{{.State.Status}}", CONTAINER],
    capture_output=True, text=True,
)
state = out.stdout.strip()
record("docker container state", state == "running", state or "<not found>")

# État du healthcheck
out = subprocess.run(
    ["docker", "inspect", "-f", "{{.State.Health.Status}}", CONTAINER],
    capture_output=True, text=True,
)
health = out.stdout.strip()
record("docker healthcheck", health == "healthy", health or "<no healthcheck>")

# Uptime container (utile pour debug : un container restarted < 90s n'a pas fini son login)
out = subprocess.run(
    ["docker", "inspect", "-f", "{{.State.StartedAt}}", CONTAINER],
    capture_output=True, text=True,
)
started_at = out.stdout.strip()
print(f"  [INFO] StartedAt = {started_at}")

== container state + healthcheck ==
  [OK  ] docker container state  | running
  [FAIL] docker healthcheck  | unhealthy
  [INFO] StartedAt = 2026-04-28T08:09:00.122826273Z


## 2. TCP probe `127.0.0.1:4002` depuis l'host

**Ce que tu dois voir** : `connect()` socket TCP réussit en < 1s. C'est ce que fait le healthcheck du container (`nc -z 127.0.0.1 4002`) côté **interne**, ici on le rejoue côté **host** pour valider que le port-mapping `127.0.0.1:4002:4002` fonctionne.

Si ça fail : soit le container n'expose pas le port, soit Windows/WSL a déjà bind 4002 (vérifier avec `Test-NetConnection -ComputerName localhost -Port 4002`).

In [ ]:
print("== TCP probe host -> 127.0.0.1:4002 ==")

t0 = time.perf_counter()
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(2.0)
try:
    sock.connect((HOST, PORT))
    dt_ms = (time.perf_counter() - t0) * 1000
    record("TCP connect 127.0.0.1:4002", True, f"{dt_ms:.1f} ms")
except (ConnectionRefusedError, socket.timeout, OSError) as e:
    record("TCP connect 127.0.0.1:4002", False, f"{type(e).__name__}: {e}")
finally:
    sock.close()

## 3. Secrets présents en env (sans exposition)

**Règle absolue** (cf. `CLAUDE.md` § « zéro exposition des secrets ») : on **ne lit jamais** la valeur d'un secret. On vérifie uniquement `bool(os.environ.get(KEY))` et `len(...)` pour confirmer qu'il est non-vide.

**Ce que tu dois voir** : les trois secrets présents avec une longueur > 0. Si l'un est manquant → relancer `.\scripts\load_secrets.ps1` dans la session PowerShell qui a lancé Jupyter.

In [ ]:
print("== secrets en env (length-only check) ==")

for key in ("IB_USERID", "IB_PASSWORD", "VNC_PASSWORD"):
    val = os.environ.get(key, "")
    record(f"{key} set", bool(val), f"length = {len(val)}" if val else "MISSING")

## 4. `IB.connect()` + handshake TWS API

**Ce que tu dois voir** : `connect()` rend la main en < 5s, `isConnected() == True`, `client.serverVersion()` retourne un int (typiquement 176+ pour Gateway 10.x).

**Si ça hang** : le TCP probe (§ 2) a réussi mais TWS API n'est pas prête → IBC est encore au login. Attendre, ou relire les logs : `docker logs fxvol-ib-gateway --tail 30 | grep -i 'login\|connected'`.

**Si erreur 326 (`client id already in use`)** : un autre process (l'app PyQt v1 ? un autre notebook ?) tient déjà clientId 199 sur cette session Gateway. Bumper `CLIENT_ID` à 200 ou 201.

In [ ]:
print("== IB.connect() + handshake ==")

t0 = time.perf_counter()
try:
    ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=15)
    dt = time.perf_counter() - t0
    record("IB.connect()", ib.isConnected(), f"{dt*1000:.0f} ms")
except Exception as e:
    record("IB.connect()", False, f"{type(e).__name__}: {e}")
    raise

# Server version (pas un secret — c'est la version du protocole TWS API)
sv = ib.client.serverVersion()
record("serverVersion", isinstance(sv, int) and sv >= 176, f"v{sv}")

# Connection time (string renvoyée par TWS au handshake)
ct = ib.client.twsConnectionTime()
record("twsConnectionTime", bool(ct), str(ct)[:30] if ct else "<empty>")

## 5. `reqCurrentTime()` — drift serveur vs local

**Ce que tu dois voir** : le timestamp renvoyé par TWS est dans une fenêtre de **±5 secondes** par rapport à `datetime.now(UTC)`. Un drift > 5s indique soit l'horloge host désynchronisée (NTP cassé), soit le container Docker a une horloge décalée (rare mais possible sur WSL2 après veille longue).

Pourquoi c'est critique : tous les timestamps applicatifs (snapshots, ordres, signaux) viennent du serveur IB. Si l'horloge dérive, les ordres MOC/LOC peuvent être rejetés et les corrélations multi-services (Postgres `created_at` vs IB `time`) deviennent fausses.

In [ ]:
print("== reqCurrentTime — server vs local clock ==")

server_dt = ib.reqCurrentTime()  # datetime tz-aware (UTC)
local_dt = datetime.now(timezone.utc)
drift_s = (server_dt - local_dt).total_seconds()

print(f"  server time : {server_dt.isoformat()}")
print(f"  local  time : {local_dt.isoformat()}")
print(f"  drift       : {drift_s:+.2f} s")

record("reqCurrentTime renvoie un datetime", isinstance(server_dt, datetime), type(server_dt).__name__)
record("drift |server-local| ≤ 5s", abs(drift_s) <= 5.0, f"{drift_s:+.2f}s")

## 6. `disconnect()` propre + reconnect avec autre clientId

**Ce que tu dois voir** : après `disconnect()`, `isConnected() == False`. Une nouvelle instance `IB()` peut immédiatement se reconnecter avec un **autre** clientId (198) sans erreur 326.

Pourquoi ce test : preuve qu'on n'a pas laissé de session zombie côté Gateway. Si on tente `connect()` immédiatement avec le **même** clientId 199 et que ça fail avec 326, c'est qu'IB n'a pas encore libéré la session — c'est le bug classique qui plante la reprise après crash de l'app PyQt v1. (Le test « même clientId » est volontairement absent ici : il fait l'objet du notebook `05_test_resilience.ipynb`.)

In [ ]:
print("== disconnect + reconnect avec autre clientId ==")

ib.disconnect()
record("disconnect()", not ib.isConnected(), f"isConnected={ib.isConnected()}")

ib2 = IB()
try:
    ib2.connect(HOST, PORT, clientId=CLIENT_ID_BIS, timeout=15)
    record(f"reconnect avec clientId={CLIENT_ID_BIS}", ib2.isConnected(), f"serverVersion=v{ib2.client.serverVersion()}")
finally:
    if ib2.isConnected():
        ib2.disconnect()

## Récap final

Tableau OK/FAIL + cleanup défensif (s'assurer qu'on ne laisse aucune session ouverte qui empêcherait le notebook suivant de se connecter).

In [ ]:
# Cleanup défensif
for x in (ib,):
    try:
        if x.isConnected():
            x.disconnect()
    except Exception:
        pass

n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<45} STATUS  DETAIL")
print("-" * 100)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<45} {sym:<6}  {detail}")
print("-" * 100)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\n✅ ib-gateway connection surface fully validated. Pass aux notebooks 02-06.")

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `docker container state` = `<not found>` | Le profile `ib` n'a pas été monté | `docker compose --profile ib up -d ib-gateway` |
| `healthcheck` = `starting` depuis > 2 min | IBC en attente push 2FA, ou image obsolète | `docker logs fxvol-ib-gateway --tail 100` ; valider la notif IB Key sur l'iPhone ; si `version is no longer supported` → bump l'image (cf. `infrastructure/ib-gateway/README.md`) |
| `healthcheck` = `unhealthy` | TWS API a crashé après login | Restart : `docker compose --profile ib restart ib-gateway` |
| TCP probe FAIL alors que healthcheck OK | Port-mapping `127.0.0.1:4002:4002` non actif (Windows réservé ?) | `Get-NetTCPConnection -LocalPort 4002` ; relancer compose ; sinon binder sur `4012` côté host |
| `IB_USERID`/`IB_PASSWORD`/`VNC_PASSWORD` `MISSING` | Secrets pas chargés dans la session courante | Dans la PowerShell qui a lancé Jupyter : `.\scripts\load_secrets.ps1`. Relancer le kernel Jupyter ensuite |
| `IB.connect()` hang > 15s alors que TCP OK | TWS API socket ouvert mais IBC pas fini de logger | Attendre, ou `docker logs --tail 30 fxvol-ib-gateway \| grep -i login` |
| Error 326 `client id already in use` | clientId 199 (ou 198) déjà pris par un autre process | Vérifier qu'aucun autre notebook ni l'app PyQt v1 ne tourne ; bumper `CLIENT_ID` dans la cellule setup |
| Error 502 `Couldn't connect to TWS` | TWS API pas prête, ou `READ_ONLY_API` mal configuré | Restart container ; vérifier `docker compose config \| grep READ_ONLY_API` |
| `drift` > 5s | Horloge host désynchronisée (NTP) ou WSL2 post-veille | `w32tm /resync` (Windows) ; sur WSL2, `wsl --shutdown` puis relancer Docker Desktop |
| `reconnect avec clientId=198` FAIL avec 326 | Session zombie côté Gateway | Restart container ; en prod cette situation déclenche `05_test_resilience` (test dédié) |